In [ ]:
###  LLM에게 도구를 붙인다
- LLM 은 기본적으로 질문만 하면 답변하는 모델
- 우리가 만든 함수를 골라 사용하도록 한다
- AI 에이전트의 핵심 - RAG도 마찬가지

In [ ]:
- LLM 에게 이렇게 이야기 하는 방식
- 너에게 이런저런 도구가 있다
- LLM이 필요할 때 (내가 만일 날씨 질문을 하려면 , 날씨 조회 도구 필요)
- 알아서 그 도구 선택 후 실행 -> AI 에이전트
- LLM 이 스스로 판단해서 도구를 쓰고 결과도 확인해야 최조결과를 주는 방식
#LLM이 함수를 사용하는것이 아니라 결과를 받아 사용자에게 응답하는 구조
- 작동방식
    - 함수실행은 우리가 하고, 그 결과를 다시 LLM이 받아 자연스러운 구조로 느껴지게 함

### 전체 흐름 
- LLM에게 사용 가능한 tool 목록 알려줌
- LLM이 필요한 함수를 실행하라고 요청
- 우리가 그 함수 실행
- 함수 실행 결과를 LLM에게 돌려주면 LLM이 최종 답을 만들어 반환

- LLM -> 판단
- 우리 --> 실행

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

load_dotenv() 
client = OpenAI()

In [2]:
df = pd.read_csv("../data/11-1_뉴스정제.csv")
df.head()

,제목,본문,카테고리,요약,출처URL,정제본문
0,현대백화점그룹 더현대 광주 추진,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...,경제,"6 6일 현대백화점그룹이 광주시에 문화복합몰을 만든다고 6일 밝혔으며, 광주시는 서...",https://n.news.naver.com/mnews/article/001/001...,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...
1,이스타항공 이상직 회사와 무관…오해 살 언동 말아야,전주 뉴시스 김얼 기자 이스타항공 자금 배임·횡령으로 전주교도소에 수감됐었던 이상직...,경제,이이스항공은 자금 배임·횡령으로 전주교도소에 수감됐었던 이상직 전 의원이 출소한 것...,https://n.news.naver.com/mnews/article/003/001...,전주 뉴시스 김얼 기자 이스타항공 자금 배임 횡령으로 전주교도소에 수감됐었던 이상직...
2,농협은행 농협금융 출범 10주년 기념주화 NFT 이벤트,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 ‘10주년 기념주...,경제,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 소셜미디어 인스타...,https://n.news.naver.com/mnews/article/366/000...,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 10주년 기념주화...
3,오늘부터 유류세 인하 폭 확대…하반기 바뀌는 세제·금융 정책은,img tag s 지난 30일 서울의 한 주유소. 〈사진 연합뉴스〉 img tag ...,경제,정부는 고유가 상황에 따라 국민의 유류비 부담 완화를 위해 이날부터 유류세를 법정 ...,https://n.news.naver.com/mnews/article/437/000...,img tag s 지난 30일 서울의 한 주유소 사진 연합뉴스 img tag e 오...
4,푸르덴셜생명 더 큰 드림 변액연금보험Ⅱ에 신규펀드 13종 추가,파이낸셜뉴스 푸르덴셜생명보험은 급변하는 금융시장에 대응하기 위해 무배당 더 큰 드림...,경제,지난르덴셜생명보험은 급변하는 금융시장에 대응하기 위해 무배당 더 큰 드림 변액연금보...,https://n.news.naver.com/mnews/article/014/000...,파이낸셜뉴스 푸르덴셜생명보험은 급변하는 금융시장에 대응하기 위해 무배당 더 큰 드림...
